# Svelvik CO₂ Time-Lapse Seismic Analysis

This notebook performs an **exploratory signal-level comparison** of SEG-2 cross-well seismic records acquired before, during and after the 2019 Svelvik CO₂ injection campaign.

It does not perform inversion, tomography, plume segmentation or quantitative CO₂ interpretation. Differences between surveys are treated as observations in processed records, not as proof of a causal plume effect.


## 1. Setup

Place the licensed Svelvik SEG-2 data locally under:

```text
data/
├── Baseline_data_2019/
├── CO2_injection_data_2019/
└── Post_Injection_data_2019/
```

The raw dataset is not distributed with this repository.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import matplotlib.pyplot as plt

from obspy import read
from obspy.signal.filter import envelope
from obspy.signal.trigger import classic_sta_lta, trigger_onset

warnings.filterwarnings("ignore", category=UserWarning, module="obspy.io.seg2.seg2")

DATA_DIR = Path("data")
BASELINE_DIR = DATA_DIR / "Baseline_data_2019"
INJECTION_DIR = DATA_DIR / "CO2_injection_data_2019"
POST_DIR = DATA_DIR / "Post_Injection_data_2019"

for folder in [BASELINE_DIR, INJECTION_DIR, POST_DIR]:
    if not folder.exists():
        raise FileNotFoundError(
            f"Missing {folder}. Obtain the licensed Svelvik data and place it in the expected local folder."
        )


## 2. Load traces and remove flat records

Only flat traces are removed here. This is not a general noise-rejection step.


In [ ]:
def clean_stream(folder_path):
    traces_out = []
    files = sorted(folder_path.glob("*.sg2")) + sorted(folder_path.glob("*.SG2"))
    for path in files:
        try:
            stream = read(str(path))
            for tr in stream:
                data = np.asarray(tr.data)
                if data.size and not np.allclose(data, data[0]):
                    traces_out.append(tr)
        except Exception as exc:
            print(f"Could not read {path.name}: {exc}")
    return traces_out

baseline_raw = clean_stream(BASELINE_DIR)
injection_raw = clean_stream(INJECTION_DIR)
post_raw = clean_stream(POST_DIR)

print(f"Baseline traces:       {len(baseline_raw):,}")
print(f"Injection traces:      {len(injection_raw):,}")
print(f"Post-injection traces: {len(post_raw):,}")


## 3. Match source–receiver geometry

A time-lapse comparison should use the same acquisition geometry where possible. The SEG-2 metadata are used to build a key from source line/station, receiver line/station and receiver component.

If the dataset contains repeated traces for the same key, this simple exploratory workflow keeps the first occurrence. A research workflow should explicitly handle repeats and survey geometry.


In [ ]:
def seg2_value(tr, name, default=""):
    return str(getattr(tr.stats.seg2, name, default)).strip()

def geometry_key(tr):
    return (
        seg2_value(tr, "SOURCE_LINE_NUMBER"),
        seg2_value(tr, "SOURCE_STATION_NUMBER"),
        seg2_value(tr, "RECEIVER_LINE_NUMBER"),
        seg2_value(tr, "RECEIVER_STATION_NUMBER"),
        seg2_value(tr, "RECEIVER_COMPONENT"),
    )

def first_trace_by_geometry(traces):
    out = {}
    for tr in traces:
        out.setdefault(geometry_key(tr), tr)
    return out

baseline_map = first_trace_by_geometry(baseline_raw)
injection_map = first_trace_by_geometry(injection_raw)
post_map = first_trace_by_geometry(post_raw)

common_keys = sorted(set(baseline_map) & set(injection_map) & set(post_map))
print(f"Common source–receiver geometries: {len(common_keys):,}")

if not common_keys:
    raise RuntimeError("No matching source–receiver geometry found across all three monitoring phases.")

sample_key = common_keys[0]
print("Example geometry key:", sample_key)


## 4. Inspect acquisition metadata for the matched example

Metadata differences matter. In the original exploratory run, for example, the selected baseline record had a stack count of 4 while the injection and post-injection records had stack counts of 8. Such acquisition differences are one reason not to make causal amplitude claims from the plots alone.


In [ ]:
def acquisition_summary(tr):
    return {
        "source_station": seg2_value(tr, "SOURCE_STATION_NUMBER"),
        "receiver_station": seg2_value(tr, "RECEIVER_STATION_NUMBER"),
        "component": seg2_value(tr, "RECEIVER_COMPONENT"),
        "stack": seg2_value(tr, "STACK"),
        "sample_interval": seg2_value(tr, "SAMPLE_INTERVAL"),
        "sampling_rate_hz": tr.stats.sampling_rate,
        "npts": tr.stats.npts,
        "acquisition_date": seg2_value(tr, "ACQUISITION_DATE"),
    }

for phase, tr in [
    ("Baseline", baseline_map[sample_key]),
    ("Injection", injection_map[sample_key]),
    ("Post-injection", post_map[sample_key]),
]:
    print(f"\n{phase}")
    for k, v in acquisition_summary(tr).items():
        print(f"  {k}: {v}")


## 5. Exploratory preprocessing

The original workflow used a **10–100 Hz band-pass filter** and then normalised each trace by its own maximum absolute amplitude.

That band is retained here only to make the cleaned repository consistent with the original exploratory plots. It is **not presented as a calibrated or optimal acquisition band**. For research use, inspect the survey documentation and spectrum first and justify the processing band.

Because each trace is normalised independently, the resulting plots cannot support quantitative claims about absolute signal energy or attenuation.


In [ ]:
FREQMIN = 10.0
FREQMAX = 100.0

def process_trace(tr, freqmin=FREQMIN, freqmax=FREQMAX):
    out = tr.copy()
    out.detrend("demean")
    out.taper(max_percentage=0.02, type="cosine")
    out.filter("bandpass", freqmin=freqmin, freqmax=freqmax, corners=4, zerophase=True)

    scale = np.max(np.abs(out.data))
    if scale > 0:
        out.data = out.data / scale
    return out

baseline = {k: process_trace(v) for k, v in baseline_map.items() if k in common_keys}
injection = {k: process_trace(v) for k, v in injection_map.items() if k in common_keys}
post = {k: process_trace(v) for k, v in post_map.items() if k in common_keys}


## 6. Normalised waveform and envelope comparison

These plots compare **shape and relative timing only** after filtering and per-trace normalisation.


In [ ]:
b = baseline[sample_key]
i = injection[sample_key]
p = post[sample_key]

plt.figure(figsize=(12, 4))
plt.plot(b.times(), b.data, label="Baseline")
plt.plot(i.times(), i.data, label="Injection")
plt.plot(p.times(), p.data, label="Post-injection")
plt.title("Matched-Geometry Normalised Waveform Comparison")
plt.xlabel("Time (s)")
plt.ylabel("Normalised amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

env_b = envelope(b.data)
env_i = envelope(i.data)
env_p = envelope(p.data)

plt.figure(figsize=(12, 4))
plt.plot(b.times(), env_b, label="Baseline envelope")
plt.plot(i.times(), env_i, label="Injection envelope")
plt.plot(p.times(), env_p, label="Post-injection envelope")
plt.title("Matched-Geometry Normalised Amplitude-Envelope Comparison")
plt.xlabel("Time (s)")
plt.ylabel("Normalised envelope amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 7. Illustrative automatic first-arrival pick

The STA/LTA picker below is an exploratory automatic pick. It has **not** been manually validated and should not be treated as a definitive travel-time measurement without QC.


In [ ]:
def detect_arrival(trace, nsta=100, nlta=1000, threshold_on=1.5, threshold_off=1.2):
    if len(trace.data) <= nlta:
        return None
    cft = classic_sta_lta(trace.data, nsta, nlta)
    onsets = trigger_onset(cft, threshold_on, threshold_off)
    if len(onsets) == 0:
        return None
    return float(trace.times()[onsets[0][0]])

for phase, tr in [("Baseline", b), ("Injection", i), ("Post-injection", p)]:
    arrival = detect_arrival(tr)
    print(f"{phase:14s}: {arrival if arrival is not None else 'no pick'}")


## 8. Overlay several matched geometries

The first ten common geometry keys are plotted. Each trace is independently normalised and vertically offset for visual comparison.


In [ ]:
keys_to_plot = common_keys[:10]

plt.figure(figsize=(12, 6))

phase_sets = [
    ("Baseline", baseline, "tab:blue"),
    ("Injection", injection, "tab:orange"),
    ("Post-injection", post, "tab:green"),
]

offset_step = 0.18
for phase_name, trace_map, colour in phase_sets:
    for j, key in enumerate(keys_to_plot):
        tr = trace_map[key]
        plt.plot(
            tr.times(),
            tr.data + j * offset_step,
            color=colour,
            alpha=0.75,
            label=phase_name if j == 0 else None,
        )

plt.title("Matched-Geometry Overlay: Normalised Traces")
plt.xlabel("Time (s)")
plt.ylabel("Normalised amplitude + offset")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 9. Interpretation boundary

This notebook demonstrates that baseline, injection and post-injection SEG-2 records can be loaded, matched by acquisition metadata and compared reproducibly at signal level.

It does **not** establish that a particular waveform difference was caused by the CO₂ plume. A stronger physical interpretation would require controlled repeatability/QC, justified frequency processing, geometry-aware travel-time analysis, and—depending on the research question—tomography or inversion.

No plume segmentation, inversion, quantitative saturation estimate, or causal reservoir diagnosis is performed here.


## Author

**Anuri Nwagbara**  
*Geological Engineer*
